In [ ]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

In [ ]:
import gradio as gr
from datetime import datetime
from pathlib import Path
import json
import sqlite3
import re
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
import requests
import urllib.parse

In [ ]:
SCHEMA = {
    'users': {
        'id': 'INTEGER PRIMARY KEY',
        'name': 'TEXT',
        'email': 'TEXT',
        'signup_date': 'DATE',
        'age': 'INTEGER',
        'country': 'TEXT',
        'status': 'TEXT'
    },
    'orders': {
        'id': 'INTEGER PRIMARY KEY',
        'user_id': 'INTEGER',
        'product_name': 'TEXT',
        'amount': 'REAL',
        'order_date': 'DATE',
        'status': 'TEXT'
    },
    'products': {
        'id': 'INTEGER PRIMARY KEY',
        'name': 'TEXT',
        'price': 'REAL',
        'category': 'TEXT',
        'stock': 'INTEGER'
    },
    'transactions': {
        'id': 'INTEGER PRIMARY KEY',
        'user_id': 'INTEGER',
        'amount': 'REAL',
        'date': 'DATE',
        'type': 'TEXT'
    }
}

In [ ]:
DANGEROUS_KEYWORDS = ['DROP', 'DELETE', 'ALTER', 'TRUNCATE', 'INSERT', 'UPDATE', 'EXEC', 'EXECUTE']
DANGEROUS_CHARS = [';', '--', '/*', '*/', 'xp_', 'sp_']

def is_safe(text):
    text_upper = text.upper()
    for keyword in DANGEROUS_KEYWORDS:
        if keyword in text_upper:
            return False
    for char in DANGEROUS_CHARS:
        if char in text:
            return False
    return True

In [ ]:
def get_table_name(query_text):
    query_lower = query_text.lower()

    keyword_map = {
        'users': ['user', 'users', 'customer', 'account'],
        'orders': ['order', 'purchase', 'buy'],
        'products': ['product', 'item', 'goods'],
        'transactions': ['transaction', 'payment', 'transfer']
    }

    for table, keywords in keyword_map.items():
        for keyword in keywords:
            if keyword in query_lower:
                return table

    return 'users'

In [ ]:
def extract_conditions(query_text):
    query_lower = query_text.lower()
    conditions = []

    if 'last month' in query_lower:
        conditions.append("DATE(signup_date) >= DATE('now', '-1 month')")

    if 'last week' in query_lower:
        conditions.append("DATE(date) >= DATE('now', '-7 days')")

    amount_match = re.search(r'greater than|more than|above\s+([0-9]+)', query_lower)
    if amount_match:
        conditions.append(f"amount > {amount_match.group(1)}")

    return conditions

In [ ]:
def generate_sql(query_text):

    if not is_safe(query_text):
        return None, "Unsafe query detected!"

    table = get_table_name(query_text)
    sql = f"SELECT * FROM {table}"

    conditions = extract_conditions(query_text)
    if conditions:
        sql += " WHERE " + " AND ".join(conditions)

    sql += " LIMIT 10"

    return sql, f"Generated from table '{table}'"

In [ ]:
query_history = []

def process_query(user_query, chat_history):

    if not user_query.strip():
        return chat_history

    sql_query, explanation = generate_sql(user_query)

    if sql_query is None:
        response = f"❌ {explanation}"
    else:
        response = f"```sql\n{sql_query}\n```\n\n{explanation}"
        query_history.append({
            "user": user_query,
            "sql": sql_query,
            "timestamp": datetime.now().isoformat()
        })

    chat_history.append((user_query, response))
    return chat_history

In [ ]:
with gr.Blocks(title="SQL Query Generator") as demo:

    gr.Markdown("# 📝 SQL Query Generator")

    chatbot = gr.Chatbot(height=400)

    query_input = gr.Textbox(
        placeholder="e.g. Show users who signed up last month"
    )

    send_btn = gr.Button("Generate SQL")

    send_btn.click(process_query, inputs=[query_input, chatbot], outputs=[chatbot])
    query_input.submit(process_query, inputs=[query_input, chatbot], outputs=[chatbot])

demo.launch(share=True)

/tmp/ipykernel_251/3543588283.py:5: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_251/3543588283.py:5: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0fdb8ab5d06efc8e6c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
